In [ ]:
#| default_exp latechunk_eval
#| eval: false

# Late Chunking Accuracy Eval

In [ ]:
#| eval: false
import os, certifi
os.environ.setdefault('SSL_CERT_FILE', certifi.where())
os.environ.setdefault('REQUESTS_CA_BUNDLE', certifi.where())
if os.environ.get('LITESEARCH_INSECURE_SSL') == '1':   # opt-in only; corporate MITM proxy fallback
    import requests as _r, urllib3 as _u
    _u.disable_warnings()
    _orig_send = _r.Session.send
    _r.Session.send = lambda self, *a, **k: _orig_send(self, *a, **{**k, 'verify': False})

from litesearch import database
from litesearch.utils import FastEncode, AutoLateChunkFastEncode, nomic_text_v15
from litesearch.data import chunk_spans
import numpy as np, json
from datasets import load_dataset

In [ ]:
#| eval: false
_probe = load_dataset('dwzhu/LongEmbed', 'narrativeqa')
print(_probe)
for split in _probe: print(split, _probe[split].column_names, dict(list(_probe[split][0].items())[:3]).keys())

In [ ]:
#| eval: false
TASKS = ['narrativeqa', '2wikimqa']

def load_longembed(task, max_docs=200, max_queries=100):
    'Load a LongEmbed task capped to max_docs/max_queries; keep only queries whose judged doc survives the cap.'
    ds = load_dataset('dwzhu/LongEmbed', task)
    corpus = {r['doc_id']: r['text'] for r in ds['corpus'].select(range(min(max_docs, len(ds['corpus']))))}
    queries = {r['qid']: r['text'] for r in ds['queries']}
    qrels = {}
    for r in ds['qrels']:
        qid, did = r['qid'], r['doc_id']
        if did in corpus and qid in queries: qrels.setdefault(qid, set()).add(did)
    queries = {q: queries[q] for q in list(qrels)[:max_queries]}
    qrels = {q: qrels[q] for q in queries}
    return corpus, queries, qrels

In [ ]:
#| eval: false
_corpus,_queries,_qrels = load_longembed('narrativeqa', max_docs=20, max_queries=5)
assert isinstance(_corpus, dict) and isinstance(_queries, dict) and isinstance(_qrels, dict)
assert len(_corpus) <= 20 and len(_queries) <= 5
for qid, docs in _qrels.items():
    assert qid in _queries
    assert all(d in _corpus for d in docs)
assert len(_qrels) >= 1